# IGF1R / calx-β — binding inspection (Notebook 3)

Compare per-frame MM-GBSA ΔG (Notebook 2) with ligand RMSD and receptor contacts (Notebook 1) on the same trajectory window.

**Question:** When ΔG ≥ 0, does the geometry still match the favorable-ΔG ensemble (energy fluctuation), or does it leave that ensemble (pose weakening / loss)?

**Inputs:** `md_binding_metrics.csv`; `mmpbsa/FINAL_RESULTS_MMPBSA.csv` (`Delta Energy Terms`).

**Outputs:** `binding_inspection_unfavorable_frames.csv`; `binding_inspection_summary.json`.


In [ ]:
from __future__ import annotations

import json
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_data_dir() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "md_calx_3qqu").is_dir():
            return candidate
        if (candidate / "data" / "md_calx_3qqu").is_dir():
            return candidate / "data"
    raise FileNotFoundError("Could not find data/md_calx_3qqu — run Notebooks 1 and 2 first.")


DATA_DIR = find_data_dir()
MD_ROOT = DATA_DIR / "md_calx_3qqu"
MMPBSA_DIR = MD_ROOT / "mmpbsa"
BINDING_CSV = MD_ROOT / "md_binding_metrics.csv"
RESULTS_CSV = MMPBSA_DIR / "FINAL_RESULTS_MMPBSA.csv"
SUMMARY_JSON = MD_ROOT / "binding_inspection_summary.json"
FLAGGED_CSV = MD_ROOT / "binding_inspection_unfavorable_frames.csv"

print(f"MD_ROOT={MD_ROOT}")
print(f"binding metrics: {BINDING_CSV} ({'found' if BINDING_CSV.exists() else 'MISSING'})")
print(f"MMPBSA results:  {RESULTS_CSV} ({'found' if RESULTS_CSV.exists() else 'MISSING'})")


In [ ]:
def parse_delta_energy_terms(path: Path) -> pd.DataFrame:
    """Per-frame ΔG from gmx_MMPBSA FINAL_RESULTS*.csv 'Delta Energy Terms' block."""
    if not path.exists():
        raise FileNotFoundError(path)
    lines = path.read_text().splitlines()
    start = next((i for i, ln in enumerate(lines) if ln.strip() == "Delta Energy Terms"), None)
    if start is None:
        raise ValueError(f"No 'Delta Energy Terms' block in {path}")
    block: list[str] = []
    for ln in lines[start + 1 :]:
        if not ln.strip():
            break
        if ln.endswith("Energy Terms") and not ln.startswith("Frame"):
            break
        block.append(ln)
    if len(block) < 2:
        raise ValueError(f"Empty Delta Energy Terms block in {path}")
    df = pd.read_csv(StringIO("\n".join(block)))
    if "TOTAL" not in df.columns:
        raise ValueError(f"Delta Energy Terms missing TOTAL column: {list(df.columns)}")
    return df


def infer_mmpbsa_time_ps(delta: pd.DataFrame, binding: pd.DataFrame) -> np.ndarray:
    """Place MM-GBSA frames on the Notebook 1 production time axis.

    Same frame count → use binding times. Otherwise map Frame # (or row order)
    linearly onto [t_min, t_max] from the binding metrics.
    """
    t_bind = binding["time_ps"].to_numpy(dtype=float)
    n = len(delta)
    if n == 0:
        return np.array([], dtype=float)
    if n == len(t_bind):
        return t_bind.copy()
    if n == 1:
        return np.array([float(np.median(t_bind))], dtype=float)

    if "Frame #" in delta.columns:
        frame = delta["Frame #"].to_numpy(dtype=float)
        fmin, fmax = float(frame.min()), float(frame.max())
        alpha = (frame - fmin) / (fmax - fmin) if fmax > fmin else np.linspace(0.0, 1.0, n)
    else:
        alpha = np.linspace(0.0, 1.0, n)
    return t_bind[0] + alpha * (t_bind[-1] - t_bind[0])


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not BINDING_CSV.exists():
        raise FileNotFoundError(f"Missing {BINDING_CSV}. Run Notebook 1 binding analysis first.")
    if not RESULTS_CSV.exists():
        raise FileNotFoundError(f"Missing {RESULTS_CSV}. Run Notebook 2 MM-GBSA first.")

    binding = pd.read_csv(BINDING_CSV)
    need = {"time_ps", "ligand_rmsd_nm", "receptor_contact_residues"}
    missing = need - set(binding.columns)
    if missing:
        raise ValueError(f"{BINDING_CSV} missing columns: {sorted(missing)}")
    binding = binding.sort_values("time_ps").reset_index(drop=True)

    delta = parse_delta_energy_terms(RESULTS_CSV).copy()
    delta["time_ps"] = infer_mmpbsa_time_ps(delta, binding)
    delta["delta_g_kcal"] = delta["TOTAL"].astype(float)
    delta["unfavorable"] = delta["delta_g_kcal"] >= 0.0
    return binding, delta


binding, delta = load_inputs()
print(
    f"Notebook 1 metrics: {len(binding)} frames  "
    f"({binding['time_ps'].iloc[0]:.0f}–{binding['time_ps'].iloc[-1]:.0f} ps)"
)
print(
    f"Notebook 2 ΔG:      {len(delta)} frames  "
    f"({delta['time_ps'].iloc[0]:.0f}–{delta['time_ps'].iloc[-1]:.0f} ps)  "
    f"ΔG≥0: {int(delta['unfavorable'].sum())}"
)


In [ ]:
def join_metrics(binding: pd.DataFrame, delta: pd.DataFrame) -> pd.DataFrame:
    """Nearest-neighbor join of MM-GBSA frames onto Notebook 1 geometry."""
    left = delta[["time_ps", "delta_g_kcal", "unfavorable"]].copy()
    if "Frame #" in delta.columns:
        left["mmpbsa_frame"] = delta["Frame #"].to_numpy()
    else:
        left["mmpbsa_frame"] = np.arange(1, len(delta) + 1)

    cols = ["time_ps", "ligand_rmsd_nm", "receptor_contact_residues"]
    if "receptor_contacts" in binding.columns:
        cols.append("receptor_contacts")
    right = binding[cols].copy()

    joined = pd.merge_asof(
        left.sort_values("time_ps"),
        right.sort_values("time_ps"),
        on="time_ps",
        direction="nearest",
    )
    bind_times = binding["time_ps"].to_numpy(dtype=float)
    match_idx = np.abs(joined["time_ps"].to_numpy()[:, None] - bind_times[None, :]).argmin(axis=1)
    joined["binding_time_ps"] = bind_times[match_idx]
    joined["time_match_err_ps"] = (joined["time_ps"] - joined["binding_time_ps"]).abs()
    return joined.reset_index(drop=True)


def geometry_bounds_from_favorable(joined: pd.DataFrame) -> dict[str, float]:
    """Bound-like envelope = observed range of favorable-ΔG (ΔG < 0) geometry.

    RMSD upper bound = max favorable RMSD; contact lower bound = min favorable
    contact-residue count. If fewer than two favorable frames, use the full series.
    """
    fav = joined.loc[~joined["unfavorable"]]
    if len(fav) < 2:
        ref = joined
        note = "full_series_fallback"
    else:
        ref = fav
        note = "favorable_delta_g_ensemble"

    rmsd = ref["ligand_rmsd_nm"].to_numpy(dtype=float)
    contacts = ref["receptor_contact_residues"].to_numpy(dtype=float)
    return {
        "rmsd_max_nm": float(np.max(rmsd)),
        "contacts_min": float(np.min(contacts)),
        "rmsd_median_nm": float(np.median(rmsd)),
        "contacts_median": float(np.median(contacts)),
        "n_reference_frames": int(len(ref)),
        "reference": note,
    }


def classify_row(row: pd.Series, bounds: dict[str, float]) -> str:
    like_bound = (
        float(row["ligand_rmsd_nm"]) <= bounds["rmsd_max_nm"]
        and float(row["receptor_contact_residues"]) >= bounds["contacts_min"]
    )
    if float(row["delta_g_kcal"]) < 0.0:
        return "favorable"
    return "unfavorable_energy_noise" if like_bound else "unfavorable_geometry_weak"


joined = join_metrics(binding, delta)
bounds = geometry_bounds_from_favorable(joined)
joined["class"] = joined.apply(lambda r: classify_row(r, bounds), axis=1)

bind_dt = np.diff(binding["time_ps"].to_numpy(dtype=float))
median_dt = float(np.median(bind_dt)) if len(bind_dt) else float("nan")
max_err = float(joined["time_match_err_ps"].max()) if len(joined) else 0.0

print(
    f"Geometry envelope ({bounds['reference']}): "
    f"n={bounds['n_reference_frames']}, "
    f"RMSD ≤ {bounds['rmsd_max_nm']:.3f} nm (max), "
    f"contact residues ≥ {bounds['contacts_min']:.1f} (min)"
)
print(
    f"Max |Δt| to nearest Notebook 1 frame: {max_err:.2f} ps "
    f"(median binding Δt = {median_dt:.2f} ps)"
)
joined.head()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
bad = joined[joined["unfavorable"]]

ax = axes[0]
ax.plot(joined["time_ps"], joined["delta_g_kcal"], lw=1.0, color="#377eb8")
ax.axhline(0.0, color="0.4", ls=":", lw=1)
mean_dg = float(joined["delta_g_kcal"].mean())
ax.axhline(mean_dg, color="#e41a1c", ls="--", lw=1, label=f"mean = {mean_dg:.1f}")
if not bad.empty:
    ax.scatter(
        bad["time_ps"], bad["delta_g_kcal"], s=18, c="#e41a1c", zorder=3,
        label=f"ΔG ≥ 0 ({len(bad)})",
    )
ax.set_ylabel("ΔG (kcal/mol)")
ax.set_title("MM-GBSA ΔG with Notebook 1 geometry")
ax.legend(loc="best", fontsize=8)

ax = axes[1]
ax.plot(binding["time_ps"], binding["ligand_rmsd_nm"], lw=1.0, color="#4daf4a")
ax.axhline(
    bounds["rmsd_max_nm"], color="0.3", ls="--", lw=1,
    label=f"favorable max = {bounds['rmsd_max_nm']:.3f} nm",
)
if not bad.empty:
    ax.scatter(bad["time_ps"], bad["ligand_rmsd_nm"], s=18, c="#e41a1c", zorder=3)
ax.set_ylabel("Ligand RMSD (nm)")
ax.legend(loc="best", fontsize=8)

ax = axes[2]
ax.plot(binding["time_ps"], binding["receptor_contact_residues"], lw=1.0, color="#984ea3")
ax.axhline(
    bounds["contacts_min"], color="0.3", ls="--", lw=1,
    label=f"favorable min = {bounds['contacts_min']:.1f}",
)
if not bad.empty:
    ax.scatter(bad["time_ps"], bad["receptor_contact_residues"], s=18, c="#e41a1c", zorder=3)
ax.set_ylabel("Contact residues")
ax.set_xlabel("Time (ps)")
ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

if not bad.empty:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].scatter(joined["ligand_rmsd_nm"], joined["delta_g_kcal"], s=10, alpha=0.35, c="#377eb8")
    axes[0].scatter(bad["ligand_rmsd_nm"], bad["delta_g_kcal"], s=22, c="#e41a1c", label="ΔG ≥ 0")
    axes[0].axhline(0, color="0.4", ls=":")
    axes[0].axvline(bounds["rmsd_max_nm"], color="0.4", ls="--")
    axes[0].set_xlabel("Ligand RMSD (nm)")
    axes[0].set_ylabel("ΔG (kcal/mol)")
    axes[0].legend(fontsize=8)

    axes[1].scatter(
        joined["receptor_contact_residues"], joined["delta_g_kcal"],
        s=10, alpha=0.35, c="#377eb8",
    )
    axes[1].scatter(
        bad["receptor_contact_residues"], bad["delta_g_kcal"],
        s=22, c="#e41a1c", label="ΔG ≥ 0",
    )
    axes[1].axhline(0, color="0.4", ls=":")
    axes[1].axvline(bounds["contacts_min"], color="0.4", ls="--")
    axes[1].set_xlabel("Receptor contact residues")
    axes[1].set_ylabel("ΔG (kcal/mol)")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
unfav = joined[joined["unfavorable"]].copy()
n_frames = len(joined)
n_unfav = len(unfav)
n_noise = int((unfav["class"] == "unfavorable_energy_noise").sum()) if n_unfav else 0
n_geo = int((unfav["class"] == "unfavorable_geometry_weak").sum()) if n_unfav else 0

print("Class counts:")
for k, v in sorted(joined["class"].value_counts().items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {k}: {v}")

if n_unfav:
    cols = [
        c for c in [
            "mmpbsa_frame", "time_ps", "delta_g_kcal", "ligand_rmsd_nm",
            "receptor_contact_residues", "receptor_contacts", "class", "time_match_err_ps",
        ]
        if c in unfav.columns
    ]
    display(Markdown(f"### Frames with ΔG ≥ 0: **{n_unfav} / {n_frames}**"))
    display(unfav[cols].sort_values("time_ps").reset_index(drop=True))
    print(f"Among ΔG≥0: energy noise = {n_noise}; geometry weak = {n_geo}")
else:
    display(Markdown(f"**No frames with ΔG ≥ 0** ({n_frames} MM-GBSA frames)."))


In [ ]:
def build_interpretation(joined: pd.DataFrame, bounds: dict[str, float]) -> str:
    """Final scientific reading for this trajectory window."""
    n = len(joined)
    unfav = joined[joined["unfavorable"]]
    n_bad = len(unfav)
    mean_dg = float(joined["delta_g_kcal"].mean())
    std_dg = float(joined["delta_g_kcal"].std(ddof=1)) if n > 1 else 0.0
    med_rmsd = float(joined["ligand_rmsd_nm"].median())
    med_c = float(joined["receptor_contact_residues"].median())
    t0, t1 = float(joined["time_ps"].iloc[0]), float(joined["time_ps"].iloc[-1])
    ref_label = bounds["reference"].replace("_", " ")

    lines = [
        (
            f"Over **{t0:.0f}–{t1:.0f} ps** ({n} MM-GBSA frames), mean ΔG = "
            f"**{mean_dg:.2f} ± {std_dg:.2f} kcal/mol**. "
            f"Median ligand RMSD = **{med_rmsd:.3f} nm**; median receptor contact residues = "
            f"**{med_c:.1f}**."
        ),
        (
            f"Bound-like geometry envelope from the **{ref_label}** "
            f"(n = {bounds['n_reference_frames']}): "
            f"RMSD ≤ **{bounds['rmsd_max_nm']:.3f} nm** (observed max) and "
            f"contact residues ≥ **{bounds['contacts_min']:.1f}** (observed min)."
        ),
    ]

    if n_bad == 0:
        lines.append(
            "**Conclusion:** All MM-GBSA frames have ΔG < 0, and pose metrics remain inside "
            "one continuous bound-like window. On this trajectory the complex is energetically "
            "and geometrically consistent with a retained calx-β / IGF1R pocket pose. "
            "MM-GBSA values here are relative free-energy estimates on simulated snapshots, "
            "not an experimental dissociation constant."
        )
        return "\n\n".join(lines)

    n_noise = int((unfav["class"] == "unfavorable_energy_noise").sum())
    n_geo = int((unfav["class"] == "unfavorable_geometry_weak").sum())
    frac = 100.0 * n_bad / n
    q75 = float(joined["time_ps"].quantile(0.75))
    n_late = int((unfav["time_ps"] >= q75).sum())
    frac_late = 100.0 * n_late / n_bad

    fav = joined.loc[~joined["unfavorable"]]
    med_rmsd_bad = float(unfav["ligand_rmsd_nm"].median())
    med_c_bad = float(unfav["receptor_contact_residues"].median())
    if len(fav):
        geo_cmp = (
            f"Median geometry on ΔG≥0 frames: RMSD **{med_rmsd_bad:.3f} nm**, "
            f"contacts **{med_c_bad:.1f}** "
            f"(vs favorable medians **{float(fav['ligand_rmsd_nm'].median()):.3f} nm**, "
            f"**{float(fav['receptor_contact_residues'].median()):.1f}**)."
        )
    else:
        geo_cmp = (
            f"Median geometry on ΔG≥0 frames: RMSD **{med_rmsd_bad:.3f} nm**, "
            f"contacts **{med_c_bad:.1f}** (no favorable-ΔG frames for comparison)."
        )

    lines.append(
        f"**{n_bad}/{n}** frames ({frac:.1f}%) have ΔG ≥ 0: "
        f"**{n_noise}** remain inside the favorable geometry envelope (energy noise), "
        f"**{n_geo}** fall outside it (geometry weak relative to the favorable ensemble). "
        f"{geo_cmp} "
        f"**{n_late}/{n_bad}** ({frac_late:.0f}%) of unfavorable frames fall in the last time "
        f"quartile (t ≥ {q75:.0f} ps)."
    )

    if n_noise >= n_geo:
        lines.append(
            "**Conclusion:** Unfavorable ΔG frames are predominantly **energy noise**: ligand RMSD "
            "and pocket contacts stay inside the favorable-ΔG geometric envelope, and mean ΔG "
            "remains negative. On this trajectory the results support a **retained bound pose** "
            "with intermittent MM-GBSA fluctuations above zero, not systematic dissociation. "
            "This is not an experimental Kd."
        )
    else:
        lines.append(
            "**Conclusion:** Most ΔG ≥ 0 frames also leave the favorable geometric envelope "
            "(higher RMSD and/or fewer contact residues). On this trajectory, unfavorable energies "
            "coincide with **weakened pocket geometry** and are not explained by energy fluctuation "
            "alone; a stably retained docked pose is **not** supported for the full window. "
            "This is not an experimental Kd."
        )

    return "\n\n".join(lines)


interpretation = build_interpretation(joined, bounds)
display(Markdown("## Interpretation and conclusion\n\n" + interpretation))

unfav = joined[joined["unfavorable"]].copy()
unfav.to_csv(FLAGGED_CSV, index=False)

summary = {
    "n_mmpbsa_frames": int(len(joined)),
    "time_ps_start": float(joined["time_ps"].iloc[0]) if len(joined) else None,
    "time_ps_end": float(joined["time_ps"].iloc[-1]) if len(joined) else None,
    "n_unfavorable_dg": int(unfav.shape[0]),
    "n_energy_noise": int((unfav["class"] == "unfavorable_energy_noise").sum()) if len(unfav) else 0,
    "n_geometry_weak": int((unfav["class"] == "unfavorable_geometry_weak").sum()) if len(unfav) else 0,
    "mean_delta_g_kcal": float(joined["delta_g_kcal"].mean()),
    "std_delta_g_kcal": float(joined["delta_g_kcal"].std(ddof=1)) if len(joined) > 1 else 0.0,
    "median_ligand_rmsd_nm": float(joined["ligand_rmsd_nm"].median()),
    "median_contact_residues": float(joined["receptor_contact_residues"].median()),
    "class_counts": {k: int(v) for k, v in joined["class"].value_counts().items()},
    "geometry_bounds_inferred": bounds,
    "max_time_match_err_ps": float(joined["time_match_err_ps"].max()) if len(joined) else None,
    "paths": {
        "binding_metrics": str(BINDING_CSV),
        "mmpbsa_csv": str(RESULTS_CSV),
        "flagged_frames_csv": str(FLAGGED_CSV),
    },
    "interpretation": interpretation,
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2) + "\n")
print(f"Wrote {FLAGGED_CSV}")
print(f"Wrote {SUMMARY_JSON}")
